<a href="https://colab.research.google.com/github/cindysocialnova/thesis_research_assistant_RAG/blob/main/thesis_research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install groq langchain_community

In [6]:
pip install groq

In [7]:
pip install langchain_community

In [9]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import gradio as gr
import requests
import pandas as pd
import re
from datetime import datetime
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize Embedding Layer
print("Loading Embedding Layer...")
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except Exception:
    pass

groq_client = Groq()

def clean_query_for_api(query_text):
    """Strips conversational filler if the LLM ignores instructions."""
    # Remove common conversational intros
    filler_phrases = ["I will search for", "Here is a query:", "Searching for", "However, I don't see", "I cannot find", "Reasoning:", "Search Query:"]
    temp_q = query_text
    for phrase in filler_phrases:
        temp_q = re.sub(phrase, "", temp_q, flags=re.IGNORECASE)
    # Remove punctuation that breaks API like quotes
    temp_q = re.sub(r'[^\w\s\-]', '', temp_q)
    return temp_q.strip()

def fetch_real_arxiv_papers(search_query, max_results=6):
    base_url = "http://export.arxiv.org/api/query"
    clean_q = clean_query_for_api(search_query)
    if not clean_q: return []

    params = {"search_query": f"all:{clean_q}", "max_results": max_results, "sortBy": "relevance"}
    try:
        response = requests.get(base_url, params=params, timeout=30)
        if response.status_code != 200: return []
        root = ET.fromstring(response.content)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        extracted = []
        for entry in root.findall('atom:entry', ns):
            extracted.append({
                "title": entry.find('atom:title', ns).text.strip().replace('\n', ' '),
                "abstract": entry.find('atom:summary', ns).text.strip().replace('\n', ' '),
                "url": entry.find('atom:id', ns).text.strip()
            })
        return extracted
    except Exception: return []

def call_production_llm(prompt):
    try:
        completion = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.1-8b-instant",
            temperature=0.3,
            max_tokens=1024,
        )
        return completion.choices[0].message.content.strip()
    except Exception as e: return f"LLM Error: {str(e)}"

def _get_thesis_analysis(user_thesis):
    analysis_prompt = (
        f"Analyze the following thesis statement: '{user_thesis}'.\n"
        "1. Determine the overall sentiment of the thesis (Positive, Negative, Neutral) towards its main subject.\n"
        "2. Identify if the thesis is framed to be supported or to be disproven/challenged.\n"
        "3. If the thesis is framed to be disproven, rephrase it to express the opposite, supporting viewpoint. Otherwise, just repeat the original thesis.\n"
        "Format your response EXACTLY as:\n"
        "Sentiment: <Sentiment>\n"
        "Stance: <Support/Disprove>\n"
        "Search_Thesis: <Rephrased or Original Thesis for search>"
    )
    analysis_output = call_production_llm(analysis_prompt)

    sentiment = "Neutral"
    stance = "Support"
    search_thesis = user_thesis

    if analysis_output:
        sentiment_match = re.search(r"Sentiment: (Positive|Negative|Neutral)", analysis_output)
        stance_match = re.search(r"Stance: (Support|Disprove)", analysis_output)
        search_thesis_match = re.search(r"Search_Thesis: (.+)", analysis_output, re.DOTALL)

        if sentiment_match:
            sentiment = sentiment_match.group(1).strip()
        if stance_match:
            stance = stance_match.group(1).strip()
        if search_thesis_match:
            search_thesis = search_thesis_match.group(1).strip()

    return sentiment, stance, search_thesis

def process_thesis_search(user_thesis):
    if len(user_thesis.strip()) < 5: return "Input too short.", "N/A", "N/A", "N/A"

    sentiment, stance, search_thesis_for_query = _get_thesis_analysis(user_thesis)

    # Use the rephrased thesis for keyword extraction if the stance is 'Disprove'
    expansion_prompt = (
        f"Identify the core scientific concepts or paper title in: '{search_thesis_for_query}'.\n"
        "Format exactly as:\nReasoning: <brief logic>\nSearch Query: <technical keywords only>"
    )

    full_output = call_production_llm(expansion_prompt)

    # Parsing Logic
    if "Search Query:" in full_output:
        reasoning_part = full_output.split("Search Query:")[0].replace("Reasoning:", "").strip()
        search_query = full_output.split("Search Query:")[-1].strip()
    else:
        reasoning_part = "LLM output was unstructured or missing Search Query. Using analyzed thesis." # Updated message
        search_query = search_thesis_for_query # Use the analyzed thesis for search if LLM fails

    # Clean the query before sending to API
    api_ready_query = clean_query_for_api(search_query)
    live_papers = fetch_real_arxiv_papers(api_ready_query)

    if not live_papers:
        # Secondary attempt: Use analyzed thesis if the LLM's keywords were too restrictive
        live_papers = fetch_real_arxiv_papers(search_thesis_for_query)
        if not live_papers:
            return f" Strategy: {reasoning_part}", f" No matches found for: {api_ready_query}", "Aborted.", f"Sentiment: {sentiment}, Stance: {stance}"

    paper_strings = [f"{p['title']}: {p['abstract']}" for p in live_papers]
    paper_embeddings = np.array(embedding_model.embed_documents(paper_strings))
    query_embedding = np.array(embedding_model.embed_query(search_thesis_for_query)) # Embed the search_thesis_for_query for score comparison
    scores = np.dot(paper_embeddings, query_embedding) / (np.linalg.norm(paper_embeddings, axis=1) * np.linalg.norm(query_embedding))

    indices = np.argsort(scores)[::-1][:5]
    paper_display_list = []
    for i in indices:
        paper_info = f"--- {live_papers[i]['title']} (Score: {scores[i]:.4f}) --- Link: {live_papers[i]['url']}"
        paper_display_list.append(paper_info)
    paper_display = "\n".join(paper_display_list)

    # The critique prompt needs to consider the original user thesis and the search strategy.
    verdict_prompt = (
        f"Original Thesis: {user_thesis}\n"
        f"Determined Sentiment: {sentiment}\n"
        f"Intended Stance: {stance}\n"
        f"Search Thesis Used (rephrased if 'Disprove'): {search_thesis_for_query}\n"
        f"Context (Top Papers):\n{paper_display}\n\n"
        "Critique the alignment of the found papers with the *original user thesis*, "
        "taking into account the 'Intended Stance'. For example, if the Stance was 'Disprove', "
        "evaluate if the papers effectively contradict the original thesis. "
        "State whether the papers generally support or contradict the ORIGINAL THESIS."
    )
    verdict = call_production_llm(verdict_prompt)

    return f" Strategy: {reasoning_part}\n Final Search String: {api_ready_query}", paper_display, verdict, f"Sentiment: {sentiment}, Stance: {stance}, Search Query Thesis: {search_thesis_for_query}"

with gr.Blocks() as demo:
    gr.Markdown("#  Thesis Agent")
    with gr.Row():
        with gr.Column():
            thesis_input = gr.Textbox(label="Thesis or Paper Title", lines=3)
            submit_btn = gr.Button("Execute", variant="primary")
        with gr.Column():
            keywords_output = gr.Textbox(label="1. Agent Strategy", interactive=False)
            sentiment_stance_output = gr.Textbox(label="1b. Thesis Analysis (Sentiment & Stance)", interactive=False) # New output
            paper_output = gr.Textbox(label="2. arXiv Matches", lines=12)
            verdict_output = gr.Textbox(label="3. Critique", lines=8)

    submit_btn.click(fn=process_thesis_search, inputs=thesis_input, outputs=[keywords_output, paper_output, verdict_output, sentiment_stance_output]) # Update outputs

demo.launch(inline=True, share=True)

Loading Embedding Layer...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://35ecf66c30da14b88c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
